In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf

import gymnasium as gym
import pickle

from VALIDATION.KeyRef_2.KeyRef_2_env2 import Luo_DDQN_env
from stable_baselines3.common.callbacks   import BaseCallback
from stable_baselines3.common.env_checker import check_env

import datetime
import pandas as pd
from stable_baselines3.common.callbacks import BaseCallback

K = 30
planning_horizon        = 480*60
ReworkProbability       = 0.03
critical_machines       = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}


purpose                 = "_tight_duedate_JA_only"
directory               = 'DATA/TIGHT_DUEDATE'

tight_duedate_setting   = True if "tight_duedate" in purpose else False
JA_only_setting         = True if "JA_only" in purpose else False

WeibullDistribution     = pd.read_excel('DATA/DataMaster.xlsx', sheet_name='Distribution')

with open('DATA/ComponentMaster.pkl', 'rb') as f:
    master = pickle.load(f)
with open(f'{directory}_VALIDATION/pickle_valid_scenarios_480.pkl', 'rb') as f:
    valid_scenarios = pickle.load(f)

env = Luo_DDQN_env(K, planning_horizon, ReworkProbability, valid_scenarios, WeibullDistribution, critical_machines,
			  	 master, tight_duedate_setting, JA_only_setting, directory)

# check_env(env)
# obs = env.reset(seed=42,
#                 test=True, 
#                 datatest="valid2", 
#                 scenariotest="A")
            
# print("Observation:", obs)

# episodes = 1
# for episode in range(episodes):
# 	done = False
# 	obs = env.reset(test=True, 
#                     datatest="valid2", 
#                     scenariotest="A")
# 	while done == False: #not done:
# 		random_action = env.action_space.sample()
# 		obs, reward, done, truncated, info = env.step(random_action)

In [2]:
import numpy as np
import torch
from stable_baselines3 import DQN

model_path = "models/LuoDDQN_loose_duedate_JA_MB-2025-01-12_01-21-19/CustomDQN_"

model = DQN.load(model_path, env=env)
def softmax_action_selection(model, obs, mu):
    obs_tensor      = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
    q_values        = model.q_net(obs_tensor).detach().cpu().numpy().flatten()
    exp_q_values    = np.exp(mu * q_values)
    probabilities   = exp_q_values / np.sum(exp_q_values)
    action          = np.random.choice(len(q_values), p=probabilities)
    return action

results = []
InstanceList = [f'valid{i+1}' for i in range(15)]
ScenarioList = ['A', 'B', 'C']

mu = 1.6 
for action_id in range(6):
    print('CDR', action_id+1)
    for instance_id in InstanceList:
        print("-----------", instance_id)
        for scenario_id in ScenarioList:
            print("-----", scenario_id)
            # Reset the environment with the new dataset
            obs, info = env.reset(test=True, 
                    datatest=instance_id, 
                    scenariotest=scenario_id,
                    fixedaction=action_id)
            
            done = False
            
            while not done:
                action = softmax_action_selection(model, obs, mu)
                obs, reward, done, truncated, info = env.step(action)
            
            tardiness = env.calc_tardiness()
            print(tardiness)
            results.append({
                            'Method'    : f'CDR{action_id+1}',
                            'InstanceID': instance_id,
                            'ScenarioID': scenario_id,
                            'Tardiness' : tardiness
                            })



Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
CDR 1
----------- valid1
----- A
0 There are 444 jobs left
0 There are 394 jobs left
0 There are 344 jobs left
0 There are 294 jobs left
0 There are 244 jobs left
0 There are 194 jobs left
0 There are 144 jobs left
0 There are 94 jobs left
0 There are 44 jobs left
find no events
find no events
find no events
find no events
find no events
find no events
find no events
====== Done ======
94610.0
----- B
0 There are 444 jobs left
0 There are 394 jobs left
0 There are 344 jobs left
0 There are 294 jobs left
0 There are 244 jobs left
0 There are 194 jobs left
0 There are 144 jobs left
0 There are 94 jobs left
0 There are 44 jobs left
find no events
find no events
find no events
find no events
find no events
find no events
find no events
====== Done ======
109885.0
----- C
0 There are 444 jobs left
0 There are 394 jobs left
0 There are 344 jobs left
0 There are 294 jobs left
0 There are 244 jobs left
0 There are 194

In [4]:
df = pd.DataFrame(results)
file_name = f"VALIDATION/LuoDDQN_{purpose}_6CDRs_FULL.xlsx"
df.to_excel(file_name, index=False)